# Notebook 05: Mechanism Split (Extension)

This tests the mechanistic prediction that Anakinra's rescue effect is selective. Since Anakinra blocks IL-1 signaling downstream of the DNA damage response, rescue is predicted to be concentrated in inflammatory/NF-κB-driven genes, while DNA-damage-response/p53 target genes are predicted to remain largely unrescued. 

The leading-edge genes from the GSEA results in notebook 03 are used for TNF-alpha Signaling via NF-kB (inflammatory) and P53 Pathway (DDR) as the two gene sets to compare.

In [1]:
import pandas as pd
from hspc_response.io import load_config, resolve_path
from hspc_response.rescue import compute_rescue_percent
from scipy import stats

## Extract gene sets from GSEA leading edges

In [2]:
config = load_config()

tables_dir = resolve_path(config["paths"]["tables_dir"])
enrichment_dir = tables_dir / "enrichment"

# Load the 96h GE_vs_RNP_NEG GSEA results, saved in notebook 03
gsea_96h_ge = pd.read_csv(enrichment_dir / "gsea_96h_GE_vs_RNP_NEG.csv")

# Pull out the leading-edge genes for the two pathways being compared
p53_row = gsea_96h_ge[gsea_96h_ge["Term"] == "p53 Pathway"]
tnf_row = gsea_96h_ge[gsea_96h_ge["Term"] == "TNF-alpha Signaling via NF-kB"]

p53_genes = p53_row["Lead_genes"].values[0].split(";")
tnf_genes = tnf_row["Lead_genes"].values[0].split(";")

print(f"P53 Pathway leading-edge genes: {len(p53_genes)}")
print(p53_genes)
print(f"\nTNF-alpha/NF-kB leading-edge genes: {len(tnf_genes)}")
print(tnf_genes)

P53 Pathway leading-edge genes: 61
['CDKN1A', 'PHLDA3', 'TNFSF9', 'RRAD', 'ZMAT3', 'RGS16', 'FDXR', 'TCHH', 'RPS27L', 'SDC1', 'ATF3', 'MDM2', 'TSPYL2', 'PLK2', 'AEN', 'DDB2', 'LIF', 'BAX', 'KLF4', 'POLH', 'HMOX1', 'ABHD4', 'SERTAD3', 'XPC', 'PDGFA', 'UPP1', 'PLK3', 'GADD45A', 'SESN1', 'PLXNB2', 'PVT1', 'CDKN2B', 'HRAS', 'DDIT4', 'RNF19B', 'RALGDS', 'IFI30', 'DDIT3', 'NOTCH1', 'ZFP36L1', 'AK1', 'TRIAP1', 'RAD9A', 'PROCR', 'IER5', 'TAX1BP3', 'PTPRE', 'SAT1', 'MXD4', 'JUN', 'CD81', 'FOS', 'DRAM1', 'JAG2', 'MXD1', 'BTG2', 'H2AJ', 'PPP1R15A', 'FGF13', 'MKNK2', 'S100A10']

TNF-alpha/NF-kB leading-edge genes: 77
['CDKN1A', 'EDN1', 'TNFSF9', 'NR4A1', 'CSF2', 'PLPP3', 'IL23A', 'CCL2', 'CXCL3', 'CCL4', 'IFIH1', 'NR4A2', 'INHBA', 'ATF3', 'TNFAIP6', 'TNFRSF9', 'TRAF1', 'DUSP5', 'KLF2', 'PLK2', 'CCL5', 'PTGS2', 'CCND1', 'BIRC3', 'LIF', 'KLF4', 'CSF1', 'EGR3', 'BCL3', 'IL7R', 'LAMB3', 'SERPINE1', 'KDM6B', 'BHLHE40', 'TNFAIP3', 'CCN1', 'DUSP4', 'ACKR3', 'BCL2A1', 'SGK1', 'GADD45A', 'PMEPA1', 'CXCL1',

## Check overlap between gene sets before comparison

In [3]:
# Check for genes appearing in both pathways' leading edges
overlap = set(p53_genes) & set(tnf_genes)
print(f"Genes in both P53 Pathway and TNF-alpha/NF-kB leading edges: {len(overlap)}")
print(sorted(overlap))

# Build exclusive gene sets for a clean mechanism comparison
p53_only = set(p53_genes) - overlap
tnf_only = set(tnf_genes) - overlap

print(f"\nP53-only genes: {len(p53_only)}")
print(f"TNF-only genes: {len(tnf_only)}")

Genes in both P53 Pathway and TNF-alpha/NF-kB leading edges: 17
['ATF3', 'BTG2', 'CDKN1A', 'DRAM1', 'FOS', 'GADD45A', 'IER5', 'JUN', 'KLF4', 'LIF', 'MXD1', 'PLK2', 'PPP1R15A', 'PTPRE', 'RNF19B', 'SAT1', 'TNFSF9']

P53-only genes: 44
TNF-only genes: 60


## Decision: use exclusive gene sets for mechanism comparison

17 genes appear in both leading-edge lists, reflecting known crosstalk between p53 and NF-κB signaling (e.g., shared stress-response effectors like CDKN1A/p21). To cleanly test whether Anakinra's rescue is selective between these two programs, we restrict the comparison to genes exclusive to each pathway: 
- 44 P53-only genes
- 60 TNF/NF-κB-only genes. 

This avoids double-counting shared genes into both categories, which would blur any selectivity signal.

## Compare rescue percentage: DDR/p53 vs. inflammatory/NF-κB genes

In [4]:

# Reload the 96h GE_vs_RNP_NEG and GE_ANAK_vs_RNP_NEG results
results_96h_ge = pd.read_csv(tables_dir / "96h_GE_vs_RNP_NEG.csv", index_col=0)
results_96h_anak = pd.read_csv(tables_dir / "96h_GE_ANAK_vs_RNP_NEG.csv", index_col=0)

# Keep only the exclusive genes that are actually present in the DE results
p53_only_in_data = [g for g in p53_only if g in results_96h_ge.index]
tnf_only_in_data = [g for g in tnf_only if g in results_96h_ge.index]

# Compute rescue percentage separately for each gene set
rescue_p53 = compute_rescue_percent(
    results_96h_ge.loc[p53_only_in_data],
    results_96h_anak,
)
rescue_tnf = compute_rescue_percent(
    results_96h_ge.loc[tnf_only_in_data],
    results_96h_anak,
)

print(f"P53-only genes (n={len(rescue_p53)}): median rescue = {rescue_p53.median():.1f}%")
print(rescue_p53.describe())
print(f"\nTNF-only genes (n={len(rescue_tnf)}): median rescue = {rescue_tnf.median():.1f}%")
print(rescue_tnf.describe())

P53-only genes (n=44): median rescue = 17.7%
count     44.000000
mean       7.692616
std       48.579302
min     -150.787315
25%      -17.963511
50%       17.687031
75%       34.267290
max      118.762636
Name: log2FoldChange, dtype: float64

TNF-only genes (n=60): median rescue = 6.1%
count     60.000000
mean       4.947871
std       66.960108
min     -168.975586
25%      -27.390530
50%        6.116613
75%       42.035278
max      212.748652
Name: log2FoldChange, dtype: float64


## Addressing numerical instability: filter to genes with a meaningful GE effect

The initial comparison showed extreme rescue percentage values (down to -169%, up to +213%), a signature of dividing by a small denominator (genes where GE itself had only a small effect). We restrict to genes with a more substantial GE effect (|log2FC| > 1, a 2-fold change) before recomputing, to ensure rescue percentages are numerically stable and interpretable.

In [5]:
effect_size_threshold = 1.0

# Restrict to genes with a stable, meaningful GE effect size (2-fold change), avoiding division-by-near-zero instability in the rescue percentage ratio
p53_stable = [g for g in p53_only_in_data if abs(results_96h_ge.loc[g, "log2FoldChange"]) > effect_size_threshold]
tnf_stable = [g for g in tnf_only_in_data if abs(results_96h_ge.loc[g, "log2FoldChange"]) > effect_size_threshold]

print(f"P53-only genes with |log2FC|>1: {len(p53_stable)} of {len(p53_only_in_data)}")
print(f"TNF-only genes with |log2FC|>1: {len(tnf_stable)} of {len(tnf_only_in_data)}")

# Recompute rescue percentage on the filtered, numerically stable gene sets
rescue_p53_stable = compute_rescue_percent(
    results_96h_ge.loc[p53_stable],
    results_96h_anak,
)
rescue_tnf_stable = compute_rescue_percent(
    results_96h_ge.loc[tnf_stable],
    results_96h_anak,
)

print(f"\nP53-only (stable, n={len(rescue_p53_stable)}): median rescue = {rescue_p53_stable.median():.1f}%")
print(rescue_p53_stable.describe())
print(f"\nTNF-only (stable, n={len(rescue_tnf_stable)}): median rescue = {rescue_tnf_stable.median():.1f}%")
print(rescue_tnf_stable.describe())

P53-only genes with |log2FC|>1: 10 of 44
TNF-only genes with |log2FC|>1: 13 of 60

P53-only (stable, n=10): median rescue = 20.4%
count    10.000000
mean     13.904196
std      18.898626
min     -17.348692
25%       3.675710
50%      20.411558
75%      27.764820
max      38.088065
Name: log2FoldChange, dtype: float64

TNF-only (stable, n=13): median rescue = 2.3%
count    13.000000
mean      5.598423
std      35.539610
min     -47.917847
25%     -25.127017
50%       2.330272
75%      24.140874
max      84.956585
Name: log2FoldChange, dtype: float64


In [6]:
# Test whether the rescue percentage distributions differ significantly between the two gene sets
statistic, p_value = stats.mannwhitneyu(rescue_p53_stable, rescue_tnf_stable, alternative="two-sided")
print(f"Mann-Whitney U test: statistic={statistic}, p-value={p_value:.4f}")

Mann-Whitney U test: statistic=82.0, p-value=0.3062


## Interpretation: mechanism split data is inconclusive

The mechanistic hypothesis predicted rescue would be concentrated in inflammatory/NF-κB genes while sparing DDR/p53 genes. After filtering to genes with a stable, meaningful GE effect size, 10 P53-only and 13 TNF-only genes remained for comparison.

Median rescue was numerically higher for P53-only genes (20.4%) than TNF-only genes (2.3%), the opposite of the original hypothesis. A Mann-Whitney U test found no significant difference between the two distributions (p=0.31).

This result is inconclusive. It does not confirm the original hypothesis, but it also does not disprove it. The small sample remaining after the effect-size filter (n=10, n=13) is substantially underpowered to detect a real difference even if one exists.

A documented biological nuance is worth noting: the original hypothesis assumed p53 and NF-κB signaling act independently, but there is documented crosstalk between them, where NF-κB signaling can reinforce and prolong p53 activation dynamics. If Anakinra disrupts this feedback loop, it could indirectly reduce the NF-κB-reinforced component of the p53 signature over time, not just NF-κB genes directly. This offers one possible explanation for the direction seen here, but it remains untested with this dataset. 